In [1]:
import random
import pandas as pd

In [ ]:
def simulate_peachs_birthday_cake(turns=20, seed=None):
    """
    Simulates a 1-player 20-turn game on Peach's Birthday Cake (Mario Party 1).
    """
    if seed is not None:
        random.seed(seed)
        
    # Define Dice Spaces on the board layout
    DICE_SPACES = {4, 11, 21, 106}
    
    records = []
    current_space = 0
    total_roll_value = 0
    skip_next_turn = False
    global_roll_number = 0
    
    def step_forward(space):
        """Moves player forward 1 space, handling board forks and loops."""
        if space == 17:
            # 1 in 4 (25%) towards Bowser's Island; 3 in 4 (75%) towards Star/Toad path
            return 101 if random.random() < 0.25 else 18
        elif space == 27 or space == 112:
            # Reaches the end of either branch -> loops back to Start (Space 0)
            return 0
        else:
            return space + 1

    def advance_player(start_pos, steps):
        pos = start_pos
        for _ in range(steps):
            pos = step_forward(pos)
        return pos

    turn = 1
    while turn <= turns:
        turn_start_space = current_space
        
        # Handle skipped turn penalty
        if skip_next_turn:
            skip_next_turn = False
            global_roll_number += 1
            records.append({
                "Turn number": turn,
                "Roll Number": global_roll_number,
                "Roll Value": 0,
                "Is Extra Roll": False,
                "Space number at the start of the turn": turn_start_space,
                "Space number at the end of the turn": current_space,
                "total roll value so far": total_roll_value
            })
            turn += 1
            continue
            
        extra_rolls = 0
        roll_in_turn = 0
        
        while True:
            global_roll_number += 1
            is_extra = (roll_in_turn > 0)
            
            # Roll standard 1-10 die
            roll_val = random.randint(1, 10)
            total_roll_value += roll_val
            
            # Track space before individual roll
            space_at_roll_start = current_space
            current_space = advance_player(current_space, roll_val)
            
            records.append({
                "Turn number": turn,
                "Roll Number": global_roll_number,
                "Roll Value": roll_val,
                "Is Extra Roll": is_extra,
                "Space number at the start of the turn": turn_start_space,
                "Space number at the end of the turn": current_space,
                "total roll value so far": total_roll_value
            })
            
            # Check landing space effect (Dice Space / Mushroom)
            if current_space in DICE_SPACES:
                if random.random() < 0.5:
                    extra_rolls += 1  # Extra roll on this turn
                else:
                    skip_next_turn = True  # Lose next turn
            
            roll_in_turn += 1
            if extra_rolls > 0:
                extra_rolls -= 1
            else:
                break
                
        turn += 1

    return pd.DataFrame(records)

In [ ]:
# Run 20-turn simulation
df_simulation = simulate_peachs_birthday_cake(turns=20, seed=42)

# Set display options to show all rows cleanly
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Display DataFrame
df_simulation